In [1]:
!pip install -q langchain langchain-openai langgraph python-dotenv flask


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
# !pip install openai python-dotenv pandas
import pandas as pd
import os, json, time
from dotenv import load_dotenv
from openai import OpenAI
import textwrap
import requests

import requests, json, textwrap

# Create a session that ignores proxy env vars (bypasses VPN for local Ollama)
SESSION = requests.Session()
SESSION.trust_env = False  # equivalent of ollama.Client(..., trust_env=False)



import truststore
truststore.inject_into_ssl()



def pretty_print(*args):
    text = " ".join(str(arg) for arg in args)
    try:
        print(textwrap.fill(text, width=80))
    except Exception as e:
        print(text)  # fallback to normal print if text is not a string

        

load_dotenv('/Users/shivam13juna/Documents/scaler/iitr_classes/llm_ref/openai_key.env')  # reads .env file in the current directory

api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    raise ValueError(
        "OPENAI_API_KEY not found! "
        "Make sure you have a .env file with: OPENAI_API_KEY=sk-..."
    )

pretty_print("API key loaded successfully.")

MODEL = 'gpt-5-nano'




client = OpenAI(api_key=api_key)
pretty_print("OpenAI client ready.")

print(f"Using model: {MODEL}")

API key loaded successfully.
OpenAI client ready.
Using model: gpt-5-nano


# LangChain

In [3]:
from langchain.chat_models import init_chat_model

# This uses the Responses API automatically!
llm = init_chat_model(f"openai:{MODEL}", api_key=api_key, temperature=0)

# Simple invocation
response = llm.invoke("Why is Donald Trump so bent on making my portfolio look so red?")
print(response.content)
print()
print(f"Model type: {type(llm).__name__}")

Short answer: he isn’t personally aiming to make your portfolio red. Markets move for lots of reasons, and a president’s actions can influence them, but they don’t set your returns directly.

Here are some likely dynamics that could explain why a portfolio might be red in this context:

- Policy signals and uncertainty: Tariffs, trade talks, tax policy, and regulatory changes can create uncertainty. Investors tend to sell riskier assets on uncertain policy outlooks, which can push stocks down in the short term.
- Sector rotations: Some Trump-era proposals or policy expectations tend to hurt certain sectors (for example, those reliant on global supply chains or foreign revenue) while benefiting others. If your holdings are concentrated in the lagging sectors, your portfolio could lag.
- Market reactions to headlines: Markets often overreact to news, then re-price as more facts come in. Short-term red days can be driven by headlines even if the long-term fundamentals are intact.
- Other 

In [ ]:
from langchain_openai import ChatOpenAI

# Explicit Responses API opt-in
llm_explicit = ChatOpenAI(model=MODEL, api_key=api_key, temperature=0, use_responses_api=True) # temperature is optional, just for demonstration not being used in responses API

response = llm_explicit.invoke("Why is Donald Trump so bent on making my portfolio look so red?")
print(response.content)

In [4]:
from langchain.messages import HumanMessage, SystemMessage, AIMessage

messages = [
    SystemMessage(content="You are a friendly NovaMart customer support agent. Keep answers brief."),
    HumanMessage(content="What's your return policy?")
]

response = llm.invoke(messages)
print(response.content)

Sure—here’s our standard policy:

- Window: 30 days from delivery
- Condition: unused, in original packaging with proof of purchase
- Eligible items: most items; non-returnable include perishables, intimate wear, software, final-sale/customized items, gift cards, etc.
- Refunds: to the original payment method
- Shipping: customer pays return shipping unless the item is defective or incorrect
- Process: start a return in your account or contact us; you’ll get a label if eligible; refunds finish 5–7 business days after we receive the item

Want me to pull the exact terms for your order? If you share your order number or the email used, I’ll check it for you.


In [5]:
messages = [
    SystemMessage(content="You are a friendly NovaMart customer support agent. Keep answers brief."),
    HumanMessage(content="What's your return policy?"),
    AIMessage(content="You can return most items within 30 days with the receipt."),
    HumanMessage(content="What if I opened the package?")
]

response = llm.invoke(messages)
print(response.content)

Opened items are returnable within 30 days as long as they’re unused and in original condition with packaging. If it’s been used or can’t be resale, it may not be eligible or could incur a restocking fee. Some items are non-returnable after opening. Want me to check the exact item for you? Please share your order number.


In [6]:
messages = [
	{"role": "system", "content": "You are a friendly NovaMart customer support agent. Keep answers brief."},
	{"role": "user", "content": "What's your return policy?"},
    {"role": "assistant", "content": "You can return most items within 30 days with the receipt."},
    {"role": "user", "content": "What if I opened the package?"}
]

response = llm.invoke(messages)

print(response.content)

Opening the package doesn’t disqualify a return. Most opened items can be returned within 30 days if they’re in new/unused condition with the original packaging and receipt. Some items may have different rules. If you’d like, share your order number and item name and I’ll confirm for that order.


# Tools in LangChain

In [ ]:
#In Openai Responses API
#1. create tool-schema
#2. create tool
#3. Provide tool to the api.

In [8]:
from langchain.tools import tool

NOVAMART_FAQ = {
    "return policy": "NovaMart offers a 30-day no-questions-asked return policy for all items in original packaging.",
    "shipping time": "Standard shipping takes 3-5 business days. Express shipping takes 1-2 business days.",
    "payment methods": "We accept Visa, Mastercard, PayPal, and NovaPay wallet.",
    "warranty": "All electronics come with a 1-year manufacturer warranty. Extended warranty available for $29/year.",
    "contact": "Email: help@novamart.com | Phone: 1-800-NOVA | Live chat available 24/7.",
}

# ----- NovaMart Order Database -----
NOVAMART_ORDERS = {
    "NM-90210": {"status": "shipped", "eta": "March 15", "item": "Wireless Headphones", "carrier": "FedEx"},
    "NM-10042": {"status": "processing", "eta": "March 18", "item": "Smart Watch Pro", "carrier": "TBD"},
    "NM-55331": {"status": "delivered", "eta": "March 10", "item": "USB-C Hub", "carrier": "UPS"},
}

@tool
def faq_lookup(topic: str) -> str:
    """Look up NovaMart FAQ by topic. Available topics: return policy, shipping time, payment methods, warranty, contact."""
    result = NOVAMART_FAQ.get(topic.lower(), None)
    if result:
        return result
    return f"No FAQ entry found for '{topic}'. Available topics: {', '.join(NOVAMART_FAQ.keys())}"



@tool
def order_tracker(order_id: str) -> str:
    """Track a NovaMart order by its ID (e.g., NM-90210)."""
    order = NOVAMART_ORDERS.get(order_id.upper(), None)
    if order:
        return f"Order {order_id}: {order['item']} — Status: {order['status']}, ETA: {order['eta']}, Carrier: {order['carrier']}"
    return f"Order {order_id} not found. Please double-check the order ID."



# Quick test — calling tools directly
print(faq_lookup.invoke({"topic": "return policy"}))
print()
print(order_tracker.invoke({"order_id": "NM-90210"}))



NovaMart offers a 30-day no-questions-asked return policy for all items in original packaging.

Order NM-90210: Wireless Headphones — Status: shipped, ETA: March 15, Carrier: FedEx


In [9]:
# Bind tools to the LLM
llm_with_tools = llm.bind_tools([faq_lookup, order_tracker])

In [10]:
response = llm_with_tools.invoke("Where is my order NM-90210?")

In [11]:
print("Content:", response.content)
print()
print("Tool calls:", response.tool_calls)

Content: 

Tool calls: [{'name': 'order_tracker', 'args': {'order_id': 'NM-90210'}, 'id': 'call_IvjjuKfD2t4zC3zzRBhGZYPU', 'type': 'tool_call'}]


# Create LangChain Agent

In [12]:
from langchain.agents import create_agent

support_agent = create_agent(
    model=f"openai:{MODEL}",
    tools=[faq_lookup, order_tracker],
    system_prompt="You are NovaMart's friendly customer support agent. Use your tools to answer customer questions accurately. Be concise and helpful."
)

print("✅ Agent created! Type:", type(support_agent).__name__)

✅ Agent created! Type: CompiledStateGraph


In [13]:
result = support_agent.invoke(
    {"messages": [{"role": "user", "content": "Hi! Can you check where my order NM-90210 is?"}]}
)

for msg in result["messages"]:
    role = msg.__class__.__name__
    content = str(msg.content)[:200] if msg.content else "(tool call)"
    print(f"[{role}] {content}")
    print()

[HumanMessage] Hi! Can you check where my order NM-90210 is?

[AIMessage] (tool call)

[ToolMessage] Order NM-90210: Wireless Headphones — Status: shipped, ETA: March 15, Carrier: FedEx

[AIMessage] Good news — NM-90210 has shipped.

- Carrier: FedEx
- Status: Shipped
- ETA: March 15

I don’t see a tracking number in this chat yet. Would you like me to pull the tracking number and share a live Fe



In [14]:
for event in support_agent.stream(
    {"messages": [{"role": "user", "content": "Tell me status of order NM-55331. I need to return order NM-55331. What's the return policy?"}]},
    stream_mode="updates"
):
    for node_name, node_output in event.items():
        print(f"--- {node_name} ---")
        if "messages" in node_output:
            for msg in node_output["messages"]:
                role = msg.__class__.__name__
                content = str(msg.content) if msg.content else "(tool call request)"
                # If this is an AI message and it requested tool calls
                if hasattr(msg, "tool_calls") and msg.tool_calls:
                    print("    Tool calls:")
                    for tc in msg.tool_calls:
                        print(f"      - tool name: {tc.get('name')}")
                        print(f"        args: {tc.get('args')}")
                        print(f"        id: {tc.get('id')}")
                print(f"  [{role}] {content}")
        print()

--- model ---
    Tool calls:
      - tool name: order_tracker
        args: {'order_id': 'NM-55331'}
        id: call_FQHp89qIu0Bc3tccHwiszVtk
      - tool name: faq_lookup
        args: {'topic': 'return policy'}
        id: call_f0teCEiRAIWqmnagH7f6PJc2
  [AIMessage] (tool call request)

--- tools ---
  [ToolMessage] Order NM-55331: USB-C Hub — Status: delivered, ETA: March 10, Carrier: UPS

--- tools ---
  [ToolMessage] NovaMart offers a 30-day no-questions-asked return policy for all items in original packaging.

--- model ---
  [AIMessage] Here you go:

- Order NM-55331 status: Delivered on March 10 via UPS (USB-C Hub).

- Return policy: 30-day no-questions-asked returns for all items in original packaging.

Since it was delivered March 10, you’re within the 30-day window. To start a return, please use your NovaMart account: Orders > NM-55331 > Return. If you’d like, I can guide you step-by-step through the return process.

